In [2]:
%pip install Whoosh

In [ ]:

import os
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID
from whoosh.qparser import QueryParser
from whoosh.scoring import TF_IDF

# Create index
schema = Schema(title=ID(stored=True), content=TEXT(stored=True))

os.makedirs("indexdir", exist_ok=True)
ix = create_in("indexdir", schema)
writer = ix.writer()

# Add documents
for file in os.listdir("docs"):
    if file.endswith(".txt"):
        with open("docs/" + file, encoding="utf-8") as f:
            writer.add_document(title=file, content=f.read())

writer.commit()

print("Documents Indexed Successfully!")

# Search documents
with ix.searcher(weighting=TF_IDF()) as s:

    query = input("Enter search keyword: ")

    parser = QueryParser("content", ix.schema)
    results = s.search(parser.parse(query))

    if not results:
        print("No documents found")

    for r in results:
        print("Document:", r["title"])
        print("Score:", round(r.score, 2))
